In [2]:
import pandas as pd
import json
import random
import numpy as np
import re

In [3]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc"
train = pd.read_csv(f"{directory_path}/train.csv")
train_summary = pd.read_csv(f"{directory_path}/training_data/training_data_grpo/train_with_questions.csv")
# train_summary = pd.read_csv(f"{directory_path}/training_data/training_data_grpo/train_summary.csv")
# test_summary = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/test_summary.csv")

/scratch/9085613.1.cds-gpu-long/ipykernel_2265204/3631891498.py:2: DtypeWarning: Columns (20,22,24,26,28,40,43,45,47,50,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,164,175,178,188,216,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,251,253,255,257,259,261,263,265,267,269,271,396,398,400,420,422,431,444,453,492,571,599,607,679,693,696,699,716,727,733,808,819,823,825,831,892,947,948,949,957,958,959,960,1017,1022,1192,1196,1199,1395,1397,1399,1400,1402,1409,1411,1413,1414,1421,1423,1425,1427,1428,1435,1450,1464,1478,1492,1494,1530,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1650,1651,1653,1654,1657,1658,1661,1662,1665,1666,1669,1670,1744,1803,1812,1814,1816,1818,1829,1831,1833,1841,1843,1845,1847,1855,1857,1859,1861,1887) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(f"{directory_path}/train.c

In [4]:
train["ID"] = train["NACCID"]
# test_summary["ID"] = test_summary["NACCID"]
train.drop(['patient_summary'], axis=1, inplace=True)

In [5]:
train_merged = train_summary.merge(train, on=['ID'])

In [6]:
train_summary['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    20966
Primary etiology    11342
MCI subtype          5756
Neuropath            2000
Amyloid PET          2000
Amyloid CSF           800
DATSCAN               200
Name: count, dtype: int64

In [7]:
# test_np = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_np.csv")
# test_np = test_np.merge(test_summary, on=["ID", "visit_summary", 'patient_summary', 'diag_summary'])

In [8]:
# test_etpr = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_etpr.csv")
# test_etpr = test_etpr.merge(test_summary, on=["ID", "visit_summary", 'patient_summary', 'diag_summary'])

In [9]:
# test_cog = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_cog.csv")
# test_cog = test_cog.merge(test_summary, on=["ID", "visit_summary", 'patient_summary', 'diag_summary'])

## Basic tests

In [8]:
# test train test overlap
def test_train_test_overlap(train, test):
    try:
        assert len(set(train['ID']).intersection(set(test['NACCID']))) == 0, "Train and test sets overlap!"
        print("✅ test_train_test_overlap: Passed")
    except AssertionError as e:
        print(f"❌ test_train_test_overlap: Failed - {e}")
        raise

    
# test_train_test_overlap(train_merged, test_summary)

In [10]:
# test for duplicate IDs
def duplicate_ids(df, column):
    try:
        assert df[column].is_unique, f"Duplicate values found in '{column}' column!"
        print(f"✅ duplicate_ids: No duplicates in '{column}' column.")
    except AssertionError as e:
        print(f"❌ duplicate_ids: {e}")
        raise

    
duplicate_ids(train_merged, "ID")
duplicate_ids(train_summary, "ID")
# duplicate_ids(test_summary, "NACCID")

✅ duplicate_ids: No duplicates in 'ID' column.
✅ duplicate_ids: No duplicates in 'ID' column.


In [11]:
# test for duplicate rows
def duplicate_rows(df):
    try:
        assert not df.duplicated().any(), "Duplicate rows found."
        print("✅ duplicate_rows: No duplicate rows found.")
    except AssertionError as e:
        print(f"❌ duplicate_rows: {e}")
        raise

    
duplicate_rows(train_merged)
duplicate_rows(train_summary)
# duplicate_rows(test_summary)

✅ duplicate_rows: No duplicate rows found.
✅ duplicate_rows: No duplicate rows found.


In [12]:
# Age test
def get_ages(df):
    ages = []
    for i, row in df.iterrows():
        pat = json.loads(row['patient_summary'])
        ages.append(pat['Subject Demographics']["Subject's age at visit"])
    return ages


def test_age_hallucinations(df):
    ages = get_ages(df)
    flag = 0
    for index, row in df.iterrows():
        text = row["visit_summary"]
        age_pattern = r"\b(\d{1,3})-year-old\s\b"
        for sentence in text.split("."):
            match = re.search(age_pattern, sentence)
            try:
                if match:  # Check if the sentence matches the pattern
                    age = match.group(1) 
                    assert int(age) == int(ages[index]), "Wrong age found."
            
            except AssertionError as e:
                flag = 1
                print(f"❌ incorrect age: {match.group(1)}. test_age_hallucinations index {index}: {e}")
             # raise
    if flag == 0:
        print("✅ test_age_hallucinations: No wrong age found.")
    
# test_age_hallucinations(train_merged)
# test_age_hallucinations(test_summary)

## Neuropath test

In [70]:
# Neuropathology Question Variants
np_question_variants = [
    "Which of the following pathologies are causing the patient's cognitive impairment based on the clinical information provided?",
    "Using the provided clinical information, determine the pathological causes of the patient's cognitive decline from the options listed below.",
    "Based on the clinical data, identify the neuropathologies responsible for the patient's cognitive impairment by selecting from the given choices.",
    "From the options below, select the disease pathologies causing cognitive impairment as indicated by the clinical information.",
    "Review the provided clinical information and choose the pathologies underlying the patient's cognitive symptoms from the available options."
]

np_dict = {
    "Alzheimer's disease pathology (AD)": {"NPADNC": [2, 3], "NACCLEWY": [0], "NPFTDTAU": [0], "NPFTDTDP": [0]},
    "Lewy body pathology (LBD)": {"NPADNC": [0, 1], "NACCLEWY": [1, 2, 3, 4], "NPFTDTAU": [0], "NPFTDTDP": [0]},
    "Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)": {"NPADNC": [0, 1], "NACCLEWY": [0]},
    "AD and LBD": {"NPADNC": [2, 3], "NACCLEWY": [1, 2, 3, 4], "NPFTDTAU": [0]},
    "AD and FTLD": {"NPADNC": [2, 3], "NACCLEWY": [0]},
    "LBD and FTLD": {"NPADNC": [0, 1], "NACCLEWY": [1, 2, 3, 4]},
    "AD and LBD and FTLD": {"NPADNC": [2, 3], "NACCLEWY": [1, 2, 3, 4]},
    "None of the above": {"NPADNC": [0, 1], "NACCLEWY": [0], "NPFTDTAU": [0]},
}


expected_np_texts = list(np_dict.keys())

In [71]:
def test_np_mapping(df, np_dict, question_variants, expected_texts):
    sub_df = df[df['Q_TYPE'] == 'Neuropath'].reset_index(drop=True)
    try:
        for i, row in sub_df.iterrows():
            # 1. Ground truth pathology logic check
            for key, valid_values in np_dict[row["ground_truth_text"]].items():
                assert int(row[key]) in valid_values, (
                    f"Row {i}: Unexpected value '{row[key]}' for key '{key}' with ground_truth_text '{row['ground_truth_text']}'"
                )
                if "FTLD" in key:
                    if row["NPFTDTAU"] == 0:
                        assert int(row["NPFTDTDP"]) == 1, (
                            f"Row {i}: Unexpected value '{row[key]}' for key '{key}' with ground_truth_text '{row['ground_truth_text']}'"
                        )
                    if row["NPFTDTDP"] == 0:
                        assert int(row["NPFTDTAU"]) == 1, (
                            f"Row {i}: Unexpected value '{row[key]}' for key '{key}' with ground_truth_text '{row['ground_truth_text']}'"
                        )
                    

            # 2. Question variant check
            assert row['question'] in question_variants, f"Row {i}: Invalid question variant."

            # 3. Options check: must contain all expected answer texts (order can vary)
            row_options = [opt.split(". ", 1)[1] for opt in row['options'].splitlines()]
            assert sorted(row_options) == sorted(expected_texts), (
                f"Row {i}: Option texts do not match expected values. \n{sorted(row_options)} \n{sorted(expected_texts)}"
            )

            # 4. Ground truth letter must point to correct text
            options_dict = {
                opt.split(". ")[0]: opt.split(". ", 1)[1] for opt in row['options'].splitlines()
            }
            assert row["ground_truth"] in options_dict, f"Row {i}: Ground truth letter not found in options."
            assert options_dict[row["ground_truth"]] == row["ground_truth_text"], (
                f"Row {i}: Ground truth letter points to incorrect text."
            )

        print("✅ test_np_mapping: All neuropath rows passed.")
    except AssertionError as e:
        print(f"❌ test_np_mapping: {e}")
        raise


In [72]:
test_np_mapping(train_merged, np_dict, np_question_variants, expected_np_texts)
# test_np_mapping(test_np, np_dict, np_question_variants, expected_np_texts)

✅ test_np_mapping: All neuropath rows passed.


In [53]:
# test_np_mapping(train_merged, np_dict, np_question_variants, np_options)
# test_np_mapping(test_np, np_dict, np_question_variants, np_options)

## Amyloid test

In [13]:
# Question variants for amyloid PET classification
pet_question_variants = [
    "Is the patient likely to have elevated amyloid pathology in the brain given the provided information?",
    "Does the patient likely have abnormal amyloid accumulation in the brain based on the clinical data?",
    "Based on the available information, is the patient likely amyloid positive in the brain?",
    "Does the evidence suggest the presence of elevated amyloid burden in the brain in this patient?",
    "Is there a high likelihood of underlying amyloid pathology in the brain based on the patient's profile?"
]

pet_dict = {
    "Yes": {"AMYLPET": 1},
    "No": {"AMYLPET": 0},
}

expected_pet_texts = list(pet_dict.keys())

In [14]:
# Keys related to amyloid biomarkers that should be removed from the patient summary
pet_keys_remove = [
    'Abnormally elevated amyloid on PET',
    'Abnormally low amyloid in CSF',
    'FDG-PET pattern of AD',
    'Tau PET evidence for AD',
    'Abnormally elevated CSF Tau or pTau'
]

In [15]:
def test_pet_mapping(df, pet_dict, question_variants, expected_texts):
    sub_df = df[df['Q_TYPE'] == 'Amyloid PET'].reset_index(drop=True)
    try:
        for i, row in sub_df.iterrows():
            # 1. Check that the question variant is valid
            assert row['question'] in question_variants, f"Row {i}: Invalid question variant."

            # 2. Extract options dictionary from the randomized options string
            options_dict = {
                opt.split(". ")[0]: opt.split(". ", 1)[1] for opt in row['options'].splitlines()
            }

            # 3. Ensure all expected texts are present (order can vary)
            actual_texts = list(options_dict.values())
            assert sorted(actual_texts) == sorted(expected_texts), (
                f"Row {i}: Option texts do not match expected values."
            )

            # 4. Check ground_truth letter is valid and maps to the correct label
            assert row['ground_truth'] in options_dict, f"Row {i}: Ground truth letter not in options."
            assert options_dict[row['ground_truth']] == row['ground_truth_text'], (
                f"Row {i}: Ground truth letter does not map to ground_truth_text."
            )

            # 5. Confirm ground_truth_text maps correctly to AMYLPET value
            assert int(row['AMYLPET']) == pet_dict[row["ground_truth_text"]]["AMYLPET"], (
                f"Row {i}: ground_truth_text '{row['ground_truth_text']}' does not match AMYLPET value."
            )

        print("✅ test_pet_mapping: All amyloid PET rows passed.")
    except AssertionError as e:
        print(f"❌ test_pet_mapping: {e}")
        raise


In [16]:
def test_pet_json_does_not_contain_keys(df, pet_keys_remove):
    sub_df = df[df['Q_TYPE'] == 'Amyloid PET'].reset_index(drop=True)
    try:
        for i, row in sub_df.iterrows():
            for key in pet_keys_remove:
                assert key not in row['patient_summary'], (
                    f"Row {i}: Found forbidden key '{key}' in patient_summary."
                )
        print("✅ test_pet_json_does_not_contain_keys: Passed")
    except AssertionError as e:
        print(f"❌ test_pet_json_does_not_contain_keys: {e}")
        raise
    
def test_pet_summary_does_not_contain_keys(df):
    flag = 0
    sub_df = df[df['Q_TYPE'] == 'Amyloid PET'].reset_index(drop=True)
    for i, row in sub_df.iterrows():
        try:
            assert "amyloid" not in row['visit_summary'].lower(), (
                f"Row {i}: Found forbidden key amyloid in visit_summary."
            )
        except AssertionError as e:
            flag = 1
            print(f"❌ test_pet_summary_does_not_contain_keys: {e}")
        # raise
    if flag == 0:
        print("✅ test_pet_summary_does_not_contain_keys: Passed")
    
    # for i, row in sub_df.iterrows():
    #     if "amyloid " in row['visit_summary'].lower():
    #         print( f"❌ Row {i}: Found forbidden key amyloid in visit_summary.")


In [17]:
test_pet_mapping(train_merged, pet_dict, pet_question_variants, expected_pet_texts)

✅ test_pet_mapping: All amyloid PET rows passed.


In [59]:
# test_pet_mapping(train_merged, pet_dict)
# test_pet_json_does_not_contain_keys(train_summary, pet_keys_remove)
# test_pet_summary_does_not_contain_keys(train_summary)

## Amyloid csf

In [18]:
# Question variants for amyloid PET classification
csf_question_variants = [
    "Is the patient likely to have elevated amyloid pathology in the brain given the provided information?",
    "Does the patient likely have abnormal amyloid accumulation in the brain based on the clinical data?",
    "Based on the available information, is the patient likely amyloid positive in the brain?",
    "Does the evidence suggest the presence of elevated amyloid burden in the brain in this patient?",
    "Is there a high likelihood of underlying amyloid pathology in the brain based on the patient's profile?"
]
csf_dict = {
    "Yes": {"AMYLCSF": 1},
    "No": {"AMYLCSF": 0},
}

expected_csf_texts = list(csf_dict.keys())

In [19]:
# Keys related to amyloid biomarkers that should be removed from the patient summary
csf_keys_remove = [
    'Abnormally elevated amyloid on PET',
    'Abnormally low amyloid in CSF',
    'FDG-PET pattern of AD',
    'Tau PET evidence for AD',
    'Abnormally elevated CSF Tau or pTau'
]

In [24]:
def test_csf_mapping(df, csf_dict, question_variants, expected_texts):
    sub_df = df[df['Q_TYPE'] == 'Amyloid CSF'].reset_index(drop=True)
    try:
        for i, row in sub_df.iterrows():
            # 1. Check that the question variant is valid
            assert row['question'] in question_variants, f"Row {i}: Invalid question variant."

            # 2. Extract options dictionary from the randomized options string
            options_dict = {
                opt.split(". ")[0]: opt.split(". ", 1)[1] for opt in row['options'].splitlines()
            }

            # 3. Ensure all expected texts are present (order can vary)
            actual_texts = list(options_dict.values())
            assert sorted(actual_texts) == sorted(expected_texts), (
                f"Row {i}: Option texts do not match expected values."
            )

            # 4. Check ground_truth letter is valid and maps to the correct label
            assert row['ground_truth'] in options_dict, f"Row {i}: Ground truth letter not in options."
            assert options_dict[row['ground_truth']] == row['ground_truth_text'], (
                f"Row {i}: Ground truth letter does not map to ground_truth_text."
            )

            # 5. Confirm ground_truth_text maps correctly to AMYLPET value
            assert int(row['AMYLCSF']) == csf_dict[row["ground_truth_text"]]["AMYLCSF"], (
                f"Row {i}: ground_truth_text '{row['ground_truth_text']}' does not match AMYLPET value."
            )

        print("✅ test_csf_mapping: All amyloid CSF rows passed.")
    except AssertionError as e:
        print(f"❌ test_pet_mapping: {e}")
        raise


In [25]:
def test_csf_json_does_not_contain_keys(df, pet_keys_remove):
    sub_df = df[df['Q_TYPE'] == 'Amyloid CSF'].reset_index(drop=True)
    try:
        for i, row in sub_df.iterrows():
            for key in pet_keys_remove:
                assert key not in row['patient_summary'], (
                    f"Row {i}: Found forbidden key '{key}' in patient_summary."
                )
        print("✅ test_pet_json_does_not_contain_keys: Passed")
    except AssertionError as e:
        print(f"❌ test_pet_json_does_not_contain_keys: {e}")
        raise
    
def test_csf_summary_does_not_contain_keys(df):
    flag = 0
    sub_df = df[df['Q_TYPE'] == 'Amyloid CSF'].reset_index(drop=True)
    for i, row in sub_df.iterrows():
        try:
            assert "amyloid" not in row['visit_summary'].lower(), (
                f"Row {i}: Found forbidden key amyloid in visit_summary."
            )
        except AssertionError as e:
            flag = 1
            print(f"❌ test_pet_summary_does_not_contain_keys: {e}")
        # raise
    if flag == 0:
        print("✅ test_pet_summary_does_not_contain_keys: Passed")
    
    # for i, row in sub_df.iterrows():
    #     if "amyloid " in row['visit_summary'].lower():
    #         print( f"❌ Row {i}: Found forbidden key amyloid in visit_summary.")


In [26]:
test_csf_mapping(train_merged, csf_dict, csf_question_variants, expected_csf_texts)

✅ test_csf_mapping: All amyloid CSF rows passed.


## Datscan test

In [27]:
# Question variants for DATScan classification
dat_question_variants = [
    "Is there evidence of Lewy body disease in the brain based on the provided information?",
    "Does the provided information suggest the presence of Lewy body disease in the brain?",
    "Is the patient likely to have Lewy body pathology in the brain based on the available data?",
    "Based on the information given, is there evidence consistent with Lewy body disease in the brain?",
    "Does the information support the presence of underlying Lewy body pathology in the brain?"
]

dat_dict = {
    "Yes": {"DATSCAN": 1},
    "No": {"DATSCAN": 0},
}

expected_dat_texts = list(dat_dict.keys())

In [28]:
# Keys related to DATScan that should be removed from the patient summary
dat_keys_remove = [
    'Dopamine transporter scan (DATscan) evidence for Lewy body disease'
]

In [29]:
def test_dat_mapping(df, dat_dict, question_variants, expected_texts):
    sub_df = df[df['Q_TYPE'] == 'DATSCAN'].reset_index(drop=True)
    try:
        for i, row in sub_df.iterrows():
            # 1. Validate question variant
            assert row['question'] in question_variants, f"Row {i}: Invalid question variant."

            # 2. Parse options string into dict {letter: text}
            options_dict = {
                opt.split(". ")[0]: opt.split(". ", 1)[1] for opt in row['options'].splitlines()
            }

            # 3. Validate all expected labels are present in any order
            actual_texts = list(options_dict.values())
            assert sorted(actual_texts) == sorted(expected_texts), (
                f"Row {i}: Option texts do not match expected values."
            )

            # 4. Ensure ground_truth letter is valid and matches correct label
            assert row['ground_truth'] in options_dict, f"Row {i}: Ground truth letter not in options."
            assert options_dict[row['ground_truth']] == row['ground_truth_text'], (
                f"Row {i}: Ground truth letter does not match ground_truth_text."
            )

            # 5. Validate that label text maps to the correct DATSCAN value
            assert int(row["DATSCAN"]) == dat_dict[row["ground_truth_text"]]["DATSCAN"], (
                f"Row {i}: ground_truth_text '{row['ground_truth_text']}' does not match DATSCAN value."
            )

        print("✅ test_dat_mapping: All DATScan rows passed.")
    except AssertionError as e:
        print(f"❌ test_dat_mapping: {e}")
        raise


In [30]:
def test_dat_json_does_not_contain_keys(df, dat_keys_remove):
    sub_df = df[df['Q_TYPE'] == 'DATSCAN'].reset_index(drop=True)
    try:
        for i, row in sub_df.iterrows():
            for key in dat_keys_remove:
                assert key not in row['patient_summary'], (
                    f"Row {i}: Found forbidden key '{key}' in patient_summary."
                )
        print("✅ test_dat_json_does_not_contain_keys: Passed")
    except AssertionError as e:
        print(f"❌ test_dat_json_does_not_contain_keys: {e}")
        raise
    
def test_dat_summary_does_not_contain_keys(df):
    sub_df = df[df['Q_TYPE'] == 'DATSCAN'].reset_index(drop=True)
    try:
        for i, row in sub_df.iterrows():
            assert "datscan" not in row['visit_summary'].lower(), (
                f"Row {i}: Found forbidden key datscan in visit_summary."
            )
        print("✅ test_dat_summary_does_not_contain_keys: Passed")
    except AssertionError as e:
        print(f"❌ test_dat_summary_does_not_contain_keys: {e}")
        raise
    
    # for i, row in sub_df.iterrows():
    #     if "datscan" in row['visit_summary'].lower():
    #         print( f"❌ Row {i}: Found forbidden key datscan in visit_summary.")


In [31]:
test_dat_mapping(train_merged, dat_dict, dat_question_variants, expected_dat_texts)

✅ test_dat_mapping: All DATScan rows passed.


In [66]:
# test_dat_mapping(train_merged, dat_dict)
# test_dat_json_does_not_contain_keys(train_summary, dat_keys_remove)
# test_dat_summary_does_not_contain_keys(train_summary)

## MCI test

In [35]:
# Question variants for MCI subtype classification
mci_question_variants = [
    "Determine, if applicable, the correct subtype of mild cognitive impairment (MCI) for this patient based on the provided information.",
    "Based on the patient's data, identify the appropriate mild cognitive impairment (MCI) subtype, if applicable.",
    "Using the information given, classify the patient's mild cognitive impairment (MCI) subtype, if applicable.",
    "If applicable, identify the subtype of mild cognitive impairment based on the provided patient details.",
    "Analyze the patient's information to identify the correct mild cognitive impairment (MCI) subtype, if applicable."
]

# Answer options for MCI subtype
mci_dict = {
    "Amnestic MCI": {"NACCTMCI": [1, 2]},
    "Non-amnestic MCI": {"NACCTMCI": [3, 4]},
    "Not applicable (no diagnosis of MCI)": {"NACCTMCI": [8]},
}

expected_mci_texts = list(mci_dict.keys())

In [36]:
def test_mci_mapping(df, mci_dict, question_variants, expected_texts):
    sub_df = df[df['Q_TYPE'] == 'MCI subtype'].reset_index(drop=True)
    try:
        for i, row in sub_df.iterrows():
            # 1. Validate question variant
            assert row['question'] in question_variants, f"Row {i}: Invalid question variant."

            # 2. Parse options string into {letter: text}
            options_dict = {
                opt.split(". ")[0]: opt.split(". ", 1)[1] for opt in row['options'].splitlines()
            }

            # 3. Confirm option texts match expected (order doesn't matter)
            actual_texts = list(options_dict.values())
            assert sorted(actual_texts) == sorted(expected_texts), (
                f"Row {i}: Option texts do not match expected values."
            )

            # 4. Ground truth letter must map to correct text
            assert row["ground_truth"] in options_dict, f"Row {i}: Ground truth letter not in options."
            assert options_dict[row["ground_truth"]] == row["ground_truth_text"], (
                f"Row {i}: Ground truth letter maps to incorrect label."
            )

            # 5. Check label maps correctly to NACCTMCI
            assert int(row["NACCTMCI"]) in mci_dict[row["ground_truth_text"]]["NACCTMCI"], (
                f"Row {i}: ground_truth_text does not match NACCTMCI value."
            )

        print("✅ test_mci_mapping: All MCI subtype rows passed.")
    except AssertionError as e:
        print(f"❌ test_mci_mapping: {e}")
        raise


In [37]:
test_mci_mapping(train_merged, mci_dict, mci_question_variants, expected_mci_texts)

✅ test_mci_mapping: All MCI subtype rows passed.


In [30]:
# test_mci_mapping(train_merged, mci_dict)

## ETPR test

In [45]:
# Question variants for primary etiology identification
etiology_question_variants = [
    "Identify the primary etiologic diagnosis of the patient using the information provided, if applicable.",
    "Identify the primary etiology of the patient's cognitive impairment using the information provided, if applicable.",
    "Based on the clinical details, determine the main cause of the patient's cognitive impairment, if applicable.",
    "Using the information provided, identify the primary cause of the patient's cognitive decline, if applicable.",
    "Determine the primary etiology underlying the patient's cognitive impairment based on the provided information, if applicable."
]

# Answer options for primary etiology prediction
etpr_dict = {
    "Alzheimer's disease (AD)": {"NACCETPR": [1]}, # AD
    "Lewy body disease (LBD)": {"NACCETPR": [2]}, # LBD
    "Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)": {"NACCETPR": [4, 5, 6, 7]}, # FTLD
    "Vascular brain injury or vascular dementia including stroke (VD)": {"NACCETPR": [8]}, # VD
    "Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)": {"NACCETPR": [17, 18, 23, 26, 27, 28, 29]}, # SEF
    "Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)": {"NACCETPR": [19, 20, 21, 22, 24, 25]},   # PSY
    # "Traumatic brain injury (TBI)": {"NACCETPR": [13]}, # TBI
    # "Normal-pressure hydrocephalus (NPH)": {"NACCETPR": [14]}, # NPH
    # "Prion disease (CJD, other)": {"NACCETPR": [12]}, # PRD
    "Other (Multiple system atrophy, Essential tremor, Down syndrome, Huntington's disease, Prion disease, Traumatic brain injury, Normal-pressure hydrocephalus, Epilepsy, CNS neoplasm, etc)": {"NACCETPR": [3, 9, 10, 11, 12, 13, 14, 15, 16]},
    "Not applicable (no cognitive impairment)": {"NACCETPR": [88]}, # Other/Unknown (default catch-all)
}

expected_etpr_texts = list(etpr_dict.keys())

In [46]:
def test_etpr_mapping(df, etpr_dict, question_variants, expected_texts):
    sub_df = df[df['Q_TYPE'] == 'Primary etiology'].reset_index(drop=True)
    try:
        for i, row in sub_df.iterrows():
            # 1. Validate question variant
            assert row['question'] in question_variants, f"Row {i}: Invalid question variant."

            # 2. Parse shuffled options into {letter: text}
            options_dict = {
                opt.split(". ")[0]: opt.split(". ", 1)[1] for opt in row['options'].splitlines()
            }

            # 3. Validate all expected options are present
            actual_texts = list(options_dict.values())
            # print(actual_texts)
            assert sorted(actual_texts) == sorted(expected_texts), (
                f"Row {i}: Option texts do not match expected values."
            )

            # 4. Check that ground_truth letter maps to the correct text
            assert row["ground_truth"] in options_dict, f"Row {i}: Ground truth letter not found in options."
            assert options_dict[row["ground_truth"]] == row["ground_truth_text"], (
                f"Row {i}: Ground truth letter maps to incorrect label."
            )

            # 5. Check that ground_truth_text maps to the correct NACCETPR value(s)
            assert int(row["NACCETPR"]) in etpr_dict[row["ground_truth_text"]]["NACCETPR"], (
                f"Row {i}: NACCETPR value does not match ground_truth_text '{row['ground_truth_text']}'."
            )

        print("✅ test_etpr_mapping: All primary etiology rows passed.")
    except AssertionError as e:
        print(f"❌ test_etpr_mapping: {e}")
        raise


In [47]:
test_etpr_mapping(train_merged, etpr_dict, etiology_question_variants, expected_etpr_texts)

✅ test_etpr_mapping: All primary etiology rows passed.


In [34]:
# test_etpr_mapping(train_merged, etpr_dict)
# test_etpr_mapping(test_etpr, etpr_dict)

## Cog status test

In [32]:
# Question variants for primary etiology identification
cog_question_variants = [
    "Using the information provided, determine the patient's cognitive status.",
    "Identify the patient's cognitive status based on the given information.",
    "Based on the provided clinical details, classify the patient's cognitive status.",
    "From the information available, determine the correct cognitive status for the patient.",
    "Analyze the patient's information and identify their cognitive status."
]

# Answer options for primary etiology prediction
cog_dict = {
    "Normal Cognition (NC)": {"NACCUDSD": 1}, 
    "Mild Cognitive Impairment (MCI)": {"NACCUDSD": 3}, 
    "Dementia (DE)": {"NACCUDSD": 4},
}

expected_cog_texts = list(cog_dict.keys())

In [33]:
def test_cog_mapping(df, cog_dict, question_variants, expected_texts):
    sub_df = df[df['Q_TYPE'] == 'Cognitive status'].reset_index(drop=True)
    try:
        for i, row in sub_df.iterrows():
            # 1. Check question variant is valid
            assert row['question'] in question_variants, f"Row {i}: Invalid question variant."

            # 2. Extract options dict from shuffled string
            options_dict = {
                opt.split(". ")[0]: opt.split(". ", 1)[1] for opt in row['options'].splitlines()
            }

            # 3. Check all expected labels are present
            actual_texts = list(options_dict.values())
            assert sorted(actual_texts) == sorted(expected_texts), (
                f"Row {i}: Option texts do not match expected values."
            )

            # 4. Check ground_truth letter maps to correct label
            assert row["ground_truth"] in options_dict, f"Row {i}: Ground truth letter not found."
            assert options_dict[row["ground_truth"]] == row["ground_truth_text"], (
                f"Row {i}: Ground truth letter maps to incorrect label."
            )

            # 5. Check that label maps correctly to NACCUDSD value
            assert int(row["NACCUDSD"]) == cog_dict[row["ground_truth_text"]]["NACCUDSD"], (
                f"Row {i}: NACCUDSD value does not match ground_truth_text."
            )

        print("✅ test_cog_mapping: All cognitive status rows passed.")
    except AssertionError as e:
        print(f"❌ test_cog_mapping: {e}")
        raise


In [34]:
test_cog_mapping(train_merged, cog_dict, cog_question_variants, expected_cog_texts)

✅ test_cog_mapping: All cognitive status rows passed.


In [38]:
# test_cog_mapping(train_merged, cog_dict)
# test_cog_mapping(test_cog, cog_dict)

## Run everything

In [73]:
def run_all_tests():
    print("========== Running All QC Tests ==========\n")
    
    # Basic structure and duplication checks
    print("train_merged")
    # test_train_test_overlap(train_merged, test_summary)
    duplicate_ids(train_merged, "ID")
    duplicate_rows(train_merged)
    print("train_summary")
    duplicate_ids(train_summary, "ID")
    duplicate_rows(train_summary)
    # print("test_summary")
    # duplicate_ids(test_summary, "NACCID")
    # duplicate_rows(test_summary)
    # print("train_merged")
    # test_age_hallucinations(train_merged)
    # print("test_summary")
    # test_age_hallucinations(test_summary)

    # Neuropathology
    print("\n--- Neuropathology ---")
    print("train_merged")
    test_np_mapping(train_merged, np_dict, np_question_variants, expected_np_texts)
    # print("test_np")
    # test_np_mapping(test_np, np_dict, np_question_variants, np_options)

    # Amyloid PET
    print("\n--- Amyloid PET ---")
    print("train_merged")
    # test_pet_mapping(train_merged, pet_dict, pet_question_variants, pet_options)
    test_pet_mapping(train_merged, pet_dict, pet_question_variants, expected_pet_texts)
    # print("train_summary")
    test_pet_json_does_not_contain_keys(train_merged, pet_keys_remove)
    # test_pet_summary_does_not_contain_keys(train_merged)

    # DATScan
    print("\n--- DATScan ---")
    print("train_merged")
    test_dat_mapping(train_merged, dat_dict, dat_question_variants, expected_dat_texts)
    # print("train_summary")
    test_dat_json_does_not_contain_keys(train_merged, dat_keys_remove)
    # test_dat_summary_does_not_contain_keys(train_merged)

    # MCI Subtype
    print("\n--- MCI Subtype ---")
    print("train_merged")
    test_mci_mapping(train_merged, mci_dict, mci_question_variants, expected_mci_texts)

    # Primary Etiology
    print("\n--- Primary Etiology ---")
    print("train_merged")
    test_etpr_mapping(train_merged, etpr_dict, etiology_question_variants, expected_etpr_texts)
    # print("test_etpr")
    # test_etpr_mapping(test_etpr, etpr_dict, etiology_question_variants, etiology_options)

    # Cognitive Status
    print("\n--- Cognitive Status ---")
    print("train_merged")
    test_cog_mapping(train_merged, cog_dict, cog_question_variants, expected_cog_texts)
    # print("test_cog")
    # test_cog_mapping(test_cog, cog_dict, cog_question_variants, cog_options)

    print("\n✅✅ All QC tests completed.\n")


In [74]:
run_all_tests()

========== Running All QC Tests ==========

train_merged
✅ duplicate_ids: No duplicates in 'ID' column.
✅ duplicate_rows: No duplicate rows found.
train_summary
✅ duplicate_ids: No duplicates in 'ID' column.
✅ duplicate_rows: No duplicate rows found.

--- Neuropathology ---
train_merged
✅ test_np_mapping: All neuropath rows passed.

--- Amyloid PET ---
train_merged
✅ test_pet_mapping: All amyloid PET rows passed.
✅ test_pet_json_does_not_contain_keys: Passed

--- DATScan ---
train_merged
✅ test_dat_mapping: All DATScan rows passed.
✅ test_dat_json_does_not_contain_keys: Passed

--- MCI Subtype ---
train_merged
✅ test_mci_mapping: All MCI subtype rows passed.

--- Primary Etiology ---
train_merged
✅ test_etpr_mapping: All primary etiology rows passed.

--- Cognitive Status ---
train_merged
✅ test_cog_mapping: All cognitive status rows passed.

✅✅ All QC tests completed.

